# PaddleOCR Test — CNIE Field Crops

Uses the already-cropped `front_cropped.jpg` (856x540).
Tests PaddleOCR with `det=False` (recognition only) on each field region.

**No preprocessing needed** — raw BGR crops go straight into the transformer model.

In [ ]:
# Run ONCE to install — PaddlePaddle FIRST, then PaddleOCR
!python -m pip install paddlepaddle==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
!python -m pip install paddleocr
!pip install shapely scikit-image "protobuf>=3.20.0,<4.0"

In [ ]:
# Verify installation
import paddle
print(f"PaddlePaddle version: {paddle.__version__}")

from paddleocr import PaddleOCR
print("PaddleOCR imported OK")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time

# Load the already-cropped card
front = cv2.imread('front_cropped.jpg')
print(f'Front loaded: {front.shape[1]}w x {front.shape[0]}h')

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(front, cv2.COLOR_BGR2RGB))
plt.title(f'Cropped Card — {front.shape[1]}x{front.shape[0]}')
plt.axis('off')
plt.show()

---
## Initialize OCR Instances

Two instances: one for Arabic, one for French/English.
Models download on first run (~10-40MB each, cached in `~/.paddleocr/`).

In [ ]:
# Initialize PaddleOCR v3.4 — PP-OCRv5 (supports Arabic, French, English)
# In v3.4, PP-OCRv4 only supports Chinese and English
# PP-OCRv5 supports Arabic + all Latin languages including French

import time

# One multilingual instance — PP-OCRv5 handles Arabic, French, English
# If lang is not specified, it auto-selects the best model

print("Loading Arabic model (PP-OCRv5)...")
t0 = time.time()
ocr_ar = PaddleOCR(lang='ar', text_rec_score_thresh=0.3)
print(f"Arabic model ready ({time.time()-t0:.1f}s)")

print("Loading French model (PP-OCRv5)...")
t0 = time.time()
ocr_fr = PaddleOCR(lang='fr', text_rec_score_thresh=0.3)
print(f"French model ready ({time.time()-t0:.1f}s)")

print("Loading English model (PP-OCRv5)...")
t0 = time.time()
ocr_en = PaddleOCR(lang='en', text_rec_score_thresh=0.3)
print(f"English model ready ({time.time()-t0:.1f}s)")

print("\nAll models loaded. Ready to OCR.")

---
## Field Definitions + OCR Helper

In [ ]:
PADDING = 10

FRONT_FIELDS = {
    "first_name_fr":      {"x": 0,   "y": 148, "w": 500, "h": 52,  "ocr": "fr"},
    "last_name_fr":       {"x": 0,   "y": 218, "w": 500, "h": 50,  "ocr": "fr"},
    "date_of_birth":      {"x": 160, "y": 255, "w": 210, "h": 54,  "ocr": "en"},
    "place_of_birth_fr":  {"x": 0,   "y": 322, "w": 500, "h": 50,  "ocr": "fr"},
    "expiry_date":        {"x": 200, "y": 365, "w": 200, "h": 45,  "ocr": "en"},
    "first_name_ar":      {"x": 330, "y": 130, "w": 270, "h": 48,  "ocr": "ar"},
    "last_name_ar":       {"x": 330, "y": 204, "w": 270, "h": 48,  "ocr": "ar"},
    "card_number":        {"x": 590, "y": 405, "w": 220, "h": 42,  "ocr": "en"},
    "gender":             {"x": 800, "y": 400, "w": 56,  "h": 45,  "ocr": "en"},
}

# Map language to OCR instance
OCR_INSTANCES = {
    "ar": ocr_ar,
    "fr": ocr_fr,
    "en": ocr_en,
}


def paddle_ocr_field(img, x, y, w, h, field_name, ocr_lang, padding=PADDING):
    """
    Crop a field region and run PaddleOCR v3.4 on it.
    Uses .predict() (v3.4 API) — .ocr() is deprecated.
    """
    ih, iw = img.shape[:2]
    x1 = max(0, x - padding)
    y1 = max(0, y - padding)
    x2 = min(iw, x + w + padding)
    y2 = min(ih, y + h + padding)

    crop = img[y1:y2, x1:x2]

    if crop.size == 0:
        print(f"ERROR: Empty crop for {field_name}")
        return None

    # Save crop temporarily — PaddleOCR v3.4 predict() works best with file paths or numpy arrays
    crop_path = f"_temp_crop_{field_name}.jpg"
    cv2.imwrite(crop_path, crop)

    # Get the right OCR instance
    ocr = OCR_INSTANCES[ocr_lang]

    # v3.4 API: use .predict() instead of deprecated .ocr()
    t0 = time.time()
    results = ocr.predict(crop_path)
    elapsed = time.time() - t0

    # Parse v3.4 result format: results is a list of OCRResult objects
    # Each result has 'rec_texts' (list of strings) and 'rec_scores' (list of floats)
    text = ""
    conf = 0.0
    if results:
        result = results[0]  # first (and only) image result
        rec_texts = result.get("rec_texts", [])
        rec_scores = result.get("rec_scores", [])
        if rec_texts:
            text = " ".join(rec_texts)
            conf = min(rec_scores) if rec_scores else 0.0

    # Clean up temp file
    import os
    if os.path.exists(crop_path):
        os.remove(crop_path)

    # Display
    plt.figure(figsize=(10, 1.5))
    plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    plt.title(f"{field_name}  |  lang={ocr_lang}  |  '{text}'  |  conf={conf:.2f}  |  {elapsed*1000:.0f}ms",
              fontsize=10, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    return {"text": text, "conf": conf, "time_ms": elapsed * 1000}


print(f"Defined {len(FRONT_FIELDS)} fields. Ready to test.")

---
## Test All Front Side Fields

In [ ]:
print(f"Testing {len(FRONT_FIELDS)} fields with PaddleOCR (det=False, raw BGR crops)")
print(f"Padding: {PADDING}px")
print("=" * 60)

all_results = {}
total_time = 0

for field_name, field in FRONT_FIELDS.items():
    result = paddle_ocr_field(
        front,
        field["x"], field["y"], field["w"], field["h"],
        field_name,
        ocr_lang=field["ocr"],
        padding=PADDING
    )
    if result:
        all_results[field_name] = result
        total_time += result["time_ms"]

# Summary
print("\n" + "=" * 60)
print("SUMMARY — PaddleOCR PP-OCRv4")
print("=" * 60)
for name, r in all_results.items():
    status = ">>" if r["conf"] > 0.5 else "!!"
    print(f"  {status} {name:20s} -> '{r['text']}'  (conf={r['conf']:.2f}, {r['time_ms']:.0f}ms)")

print(f"\nTotal OCR time: {total_time:.0f}ms for {len(all_results)} fields")
print(f"Average: {total_time/len(all_results):.0f}ms per field")

---
## Preprocessing Comparison Test

Tests 4 preprocessing variants on all fields to find what helps/hurts accuracy.

In [ ]:
import os

def preprocess_raw(img):
    return img

def preprocess_upscale2x(img):
    return cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

def preprocess_clahe(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)

def preprocess_clahe_binary(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    binary = cv2.adaptiveThreshold(enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY, 15, 4)
    return cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)

PREPROCESS_VARIANTS = {
    "raw":          {"fn": preprocess_raw,          "scale": 1},
    "upscale_2x":   {"fn": preprocess_upscale2x,    "scale": 2},
    "clahe":        {"fn": preprocess_clahe,         "scale": 1},
    "clahe_binary": {"fn": preprocess_clahe_binary,  "scale": 1},
}

print(f"Defined {len(PREPROCESS_VARIANTS)} preprocessing variants.")

In [ ]:
# Run all preprocessing variants on all fields
comparison = {}

for variant_name, variant in PREPROCESS_VARIANTS.items():
    print("=" * 70)
    print(f"VARIANT: {variant_name}")
    print("=" * 70)

    processed = variant["fn"](front)
    scale = variant["scale"]
    variant_results = {}

    for field_name, field in FRONT_FIELDS.items():
        x, y, w, h = field["x"] * scale, field["y"] * scale, field["w"] * scale, field["h"] * scale
        ih, iw = processed.shape[:2]
        pad = PADDING * scale
        x1, y1 = max(0, x - pad), max(0, y - pad)
        x2, y2 = min(iw, x + w + pad), min(ih, y + h + pad)

        crop = processed[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        crop_path = f"_temp_{variant_name}_{field_name}.jpg"
        cv2.imwrite(crop_path, crop)

        ocr = OCR_INSTANCES[field["ocr"]]
        t0 = time.time()
        results = ocr.predict(crop_path)
        elapsed = time.time() - t0

        text, conf = "", 0.0
        if results:
            r = results[0]
            texts = r.get("rec_texts", [])
            scores = r.get("rec_scores", [])
            if texts:
                text = " ".join(texts)
                conf = min(scores) if scores else 0.0

        if os.path.exists(crop_path):
            os.remove(crop_path)

        variant_results[field_name] = {"text": text, "conf": conf, "time_ms": elapsed * 1000}
        print(f"  {field_name:20s} -> '{text}'  (conf={conf:.2f})")

    comparison[variant_name] = variant_results
    print()

print("Done. Run next cell for summary table.")

In [ ]:
# Summary comparison table
variants = list(comparison.keys())
header = f"{'Field':20s} | " + " | ".join(f"{v:22s}" for v in variants)
print(header)
print("-" * len(header))

for field_name in FRONT_FIELDS:
    row = f"{field_name:20s} |"
    for v in variants:
        r = comparison[v].get(field_name, {"text": "—", "conf": 0})
        cell = f" {r['text'][:14]:14s} {r['conf']:.2f}"
        row += f" {cell:22s} |"
    print(row)

# Average confidence per variant
print("-" * len(header))
row = f"{'AVG CONFIDENCE':20s} |"
for v in variants:
    confs = [r["conf"] for r in comparison[v].values()]
    avg = sum(confs) / len(confs) if confs else 0
    cell = f" {'':14s} {avg:.2f}"
    row += f" {cell:22s} |"
print(row)

# Best variant per field
print("\nBEST VARIANT PER FIELD:")
for field_name in FRONT_FIELDS:
    best_v, best_conf = "", 0
    for v in variants:
        r = comparison[v].get(field_name, {"conf": 0})
        if r["conf"] > best_conf:
            best_conf = r["conf"]
            best_v = v
    text = comparison[best_v][field_name]["text"] if best_v else "—"
    print(f"  {field_name:20s} -> {best_v:14s} (conf={best_conf:.2f}) '{text}'")

In [ ]:
# Full Image OCR — feed the entire card to each language model
# PaddleOCR handles detection + recognition automatically

lang_models = {
    "Arabic (ar)": ocr_ar,
    "French (fr)": ocr_fr,
    "English (en)": ocr_en,
}

full_image_path = "front_cropped.jpg"

for lang_name, ocr_instance in lang_models.items():
    print("=" * 70)
    print(f"FULL IMAGE OCR — {lang_name}")
    print("=" * 70)

    t0 = time.time()
    results = ocr_instance.predict(full_image_path)
    elapsed = time.time() - t0

    if results:
        result = results[0]
        rec_texts = result.get("rec_texts", [])
        rec_scores = result.get("rec_scores", [])
        rec_polys = result.get("rec_polys", [])

        print(f"Found {len(rec_texts)} text regions in {elapsed*1000:.0f}ms\n")

        for i, (text, score) in enumerate(zip(rec_texts, rec_scores)):
            status = ">>" if score > 0.5 else "!!"
            print(f"  {status} [{i+1:2d}] conf={score:.2f}  '{text}'")

        # Draw detected regions on the image
        img_copy = front.copy()
        for poly in rec_polys:
            pts = np.array(poly, dtype=np.int32)
            cv2.polylines(img_copy, [pts], True, (0, 255, 0), 2)

        plt.figure(figsize=(14, 8))
        plt.imshow(cv2.cvtColor(img_copy, cv2.COLOR_BGR2RGB))
        plt.title(f"Full Image OCR — {lang_name} — {len(rec_texts)} regions detected")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print("No results returned.")

    print()

---
## Full Image OCR — Preprocessing Comparison

Same 4 preprocessing variants, but on the full image (detection + recognition).
Compares which preprocessing helps PaddleOCR find and read more text.

In [ ]:
# Full image OCR with preprocessing variants x language models
lang_models = {
    "Arabic (ar)": ocr_ar,
    "French (fr)": ocr_fr,
    "English (en)": ocr_en,
}

full_img_comparison = {}

for variant_name, variant in PREPROCESS_VARIANTS.items():
    processed = variant["fn"](front)
    tmp_path = f"_temp_full_{variant_name}.jpg"
    cv2.imwrite(tmp_path, processed)

    print("=" * 70)
    print(f"FULL IMAGE — {variant_name} ({processed.shape[1]}x{processed.shape[0]})")
    print("=" * 70)

    variant_results = {}

    for lang_name, ocr_instance in lang_models.items():
        t0 = time.time()
        results = ocr_instance.predict(tmp_path)
        elapsed = time.time() - t0

        texts_found = []
        if results:
            r = results[0]
            rec_texts = r.get("rec_texts", [])
            rec_scores = r.get("rec_scores", [])
            rec_polys = r.get("rec_polys", [])

            for txt, sc in zip(rec_texts, rec_scores):
                texts_found.append({"text": txt, "conf": sc})

            # Draw bounding boxes
            img_show = processed.copy()
            if len(img_show.shape) == 2:
                img_show = cv2.cvtColor(img_show, cv2.COLOR_GRAY2BGR)
            for poly in rec_polys:
                pts = np.array(poly, dtype=np.int32)
                cv2.polylines(img_show, [pts], True, (0, 255, 0), 2)

            plt.figure(figsize=(14, 8))
            plt.imshow(cv2.cvtColor(img_show, cv2.COLOR_BGR2RGB))
            plt.title(f"{variant_name} — {lang_name} — {len(rec_texts)} regions — {elapsed*1000:.0f}ms")
            plt.axis('off')
            plt.tight_layout()
            plt.show()

        print(f"\n  {lang_name}: {len(texts_found)} regions ({elapsed*1000:.0f}ms)")
        for i, t in enumerate(texts_found):
            status = ">>" if t["conf"] > 0.5 else "!!"
            print(f"    {status} [{i+1:2d}] conf={t['conf']:.2f}  '{t['text']}'")

        variant_results[lang_name] = texts_found

    full_img_comparison[variant_name] = variant_results

    if os.path.exists(tmp_path):
        os.remove(tmp_path)
    print()

# Summary: total regions detected per variant x lang
print("=" * 70)
print("SUMMARY — Regions detected per variant x language")
print("=" * 70)
print(f"{'Variant':16s} | {'Arabic':10s} | {'French':10s} | {'English':10s}")
print("-" * 55)
for v, langs in full_img_comparison.items():
    ar = len(langs.get("Arabic (ar)", []))
    fr = len(langs.get("French (fr)", []))
    en = len(langs.get("English (en)", []))
    print(f"{v:16s} | {ar:10d} | {fr:10d} | {en:10d}")

---
## Bonus: Try PP-OCRv5 (If v4 Results Are Weak)

PP-OCRv5 has ~40% accuracy improvement on some scripts. Just change `ocr_version`.

In [ ]:
# Uncomment and run this cell to test PP-OCRv5
# Only re-initialize the models that had weak results above

# print("Loading PP-OCRv5 Arabic model...")
# ocr_ar_v5 = PaddleOCR(
#     use_angle_cls=False,
#     lang='ar',
#     use_gpu=False,
#     ocr_version='PP-OCRv5',
#     drop_score=0.3,
#     show_log=False
# )
# print("Ready. Now re-run the test cell above after changing:")
# print("  OCR_INSTANCES['ar'] = ocr_ar_v5")

---
## Compare: PaddleOCR vs Tesseract

Fill in after running:

| Field | Tesseract | PaddleOCR | Winner |
|-------|-----------|-----------|--------|
| first_name_fr | ? | ? | ? |
| last_name_fr | ? | ? | ? |
| date_of_birth | ? | ? | ? |
| place_of_birth_fr | ? | ? | ? |
| expiry_date | ? | ? | ? |
| first_name_ar | ? | ? | ? |
| last_name_ar | ? | ? | ? |
| card_number | ? | ? | ? |
| gender | ? | ? | ? |

**Speed:** Tesseract ~300ms/field vs PaddleOCR ~50ms/field

**Decision:** ___